In [ ]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.merge import merge
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [7]:
GMTED2010_part1 = rio.open(DATA / 'GMTED2010_30arcsec_part1.tif')
GMTED2010_part2 = rio.open(DATA / 'GMTED2010_30arcsec_part2.tif')
GMTED2010_part3 = rio.open(DATA / 'GMTED2010_30arcsec_part3.tif')
GMTED2010_part4 = rio.open(DATA / 'GMTED2010_30arcsec_part4.tif')

In [8]:
all_parts = [GMTED2010_part1, GMTED2010_part2, GMTED2010_part3, GMTED2010_part4]

In [9]:
merged_data, merged_transform = merge(all_parts)

In [10]:
GMTED2010_1km = np.empty(country_raster_1km_.shape, dtype=np.float32)
reproject(
    source=merged_data[0],
    destination=GMTED2010_1km,
    src_transform=merged_transform,
    src_crs=GMTED2010_part4.crs,
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.nearest,
)

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(16922, 33844), dtype=float32),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [11]:
with rio.open(
    FEATURE_DIR / 'elevation-1km.tif',
    "w",
    driver="GTiff",
    height=GMTED2010_1km.shape[0],
    width=GMTED2010_1km.shape[1],
    count=1,
    dtype=GMTED2010_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(GMTED2010_1km, 1)

In [12]:
x_1km, y_1km    = np.gradient(GMTED2010_1km, 1e3, 1e3)
slope_1km       = np.arctan(np.sqrt(x_1km**2 + y_1km**2)) * (180.0 / np.pi)

In [13]:
with rio.open(
    FEATURE_DIR / 'slope-1km.tif',
    "w",
    driver="GTiff",
    height=slope_1km.shape[0],
    width=slope_1km.shape[1],
    count=1,
    dtype=slope_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(slope_1km, 1)